In [167]:
import pandas as pd

In [168]:
# Read in the data
df = pd.read_csv("../data/data.csv")

In [169]:
#clean job titles by removing special characters and converting to lowercase. keep spaces
df['job_title'] = df['job_title'].str.replace('[^a-zA-Z0-9\\s]', '', regex=True).str.lower()

#remove extra whitespace, newlines, and carriage returns from job titles
df['job_title'] = df['job_title'].str.replace(r'[\r\n\s]+', ' ', regex=True)

print(df["job_title"].value_counts())


job_title
2019 ct bauer college of business graduate magna cum laude and aspiring human resources professional                   7
aspiring human resources professional                                                                                  7
student at humber college and aspiring human resources generalist                                                      7
people development coordinator at ryan                                                                                 6
native english teacher at epik english program in korea                                                                5
aspiring human resources specialist                                                                                    5
hr senior specialist                                                                                                   5
advisory board member at celal bayar university                                                                        4
seeking human resource

In [170]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Shane\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [171]:
# join all job titles into a single string and split into words
all_titles = ' '.join(df['job_title'].dropna())
words = all_titles.split()

#remove stop words from the word counts
words_no_stopwords = [word for word in words if word not in stop_words]

def get_common_words(words, n=20):
    from collections import Counter
    word_counts = Counter(words)
    common_words = pd.DataFrame(word_counts.most_common(n), columns=['word', 'count'])
    return common_words

In [172]:
common_words_no_stopwords = get_common_words(words_no_stopwords, n=20)
common_words_no_stopwords


,word,count
0,human,63
1,resources,63
2,aspiring,35
3,professional,20
4,student,16
5,seeking,15
6,college,14
7,generalist,14
8,university,12
9,specialist,12


In [173]:
#find common phrases in job titles to the nth-gram
from sklearn.feature_extraction.text import CountVectorizer

def find_ngrams(texts, n):
    vectorizer = CountVectorizer(ngram_range=(n, n))
    
    X = vectorizer.fit_transform(texts)
    
    ngram_df = pd.DataFrame({
        'phrase': vectorizer.get_feature_names_out(),
        'count': X.sum(axis=0).A1 #A1: flatten the array to 1D
    })
    
    return ngram_df.sort_values('count', ascending=False)

In [174]:
common_phrases_2 = find_ngrams(df['job_title'].dropna(), 3)

common_phrases_2.head(50)

,phrase,count
22,aspiring human resources,35
112,human resources professional,19
14,and aspiring human,16
104,human resources generalist,10
214,seeking human resources,9
0,2019 ct bauer,7
45,business graduate magna,7
32,at humber college,7
41,bauer college of,7
53,college and aspiring,7


In [175]:
#Example list of jobs
job_title_categories = [
    "human resources",
    "finance",
    "customer service",
    "information technology",
    "data science",
    "engineering",
    "education",
    "other"
]

In [176]:
# use embeddings to find similar job titles
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# encode the job title categories
category_embeddings = model.encode(job_title_categories, normalize_embeddings=True)

# embed the job titles in the dataframe
job_title_embeddings = model.encode(df["job_title"].fillna("").tolist(), normalize_embeddings=True)

# compute cosine similarities between job titles and categories
similarities = job_title_embeddings @ category_embeddings.T

best_match_index = similarities.argmax(axis=1)
best_match_score = similarities.max(axis=1)

df["identified_job_title"] = [job_title_categories[i] for i in best_match_index]
df["job_title_similarity"] = best_match_score

df


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5662.48it/s]


,id,job_title,location,connection,fit,identified_job_title,job_title_similarity
0,1,2019 ct bauer college of business graduate mag...,"Houston, Texas",85,NaN,human resources,0.433673
1,2,native english teacher at epik english program...,Kanada,500+,NaN,education,0.329277
2,3,aspiring human resources professional,"Raleigh-Durham, North Carolina Area",44,NaN,human resources,0.773061
3,4,people development coordinator at ryan,"Denton, Texas",500+,NaN,human resources,0.441673
4,5,advisory board member at celal bayar university,"İzmir, Türkiye",500+,NaN,education,0.251772
5,6,aspiring human resources specialist,Greater New York City Area,1,NaN,human resources,0.779432
6,7,student at humber college and aspiring human r...,Kanada,61,NaN,human resources,0.601459
7,8,hr senior specialist,San Francisco Bay Area,500+,NaN,human resources,0.552003
8,9,student at humber college and aspiring human r...,Kanada,61,NaN,human resources,0.601459
9,10,seeking human resources hris and generalist po...,Greater Philadelphia Area,500+,NaN,human resources,0.724941


In [182]:
#df["location_clean"] = (df["location"].str.replace(r"\s+Area$", "", regex=True).str.strip())
df["location"] = df["location"].str.replace(r"Kanada$", "Canada", regex=True).str.strip()
has_comma = df["location"].str.contains(",", na=False)

split_location = (df.loc[has_comma, "location"].str.split(",", n=1, expand=True))

df["city"] = pd.NA
df["region"] = pd.NA
#df["country"] = pd.NA

df.loc[has_comma, "city"] = split_location[0].str.strip()
df.loc[has_comma, "region"] = split_location[1].str.strip()

is_region = ~df["location"].str.contains(",", na=False)
df.loc[is_region, "region"] = df.loc[is_region, "location"]



df

,id,job_title,location,connection,fit,identified_job_title,job_title_similarity,city,region,connections,adjusted_senior_similarity,adjusted_junior_similarity,seniority
0,1,2019 ct bauer college of business graduate mag...,"Houston, Texas",85,NaN,human resources,0.433673,Houston,Texas,85,0.323124,0.507028,junior
1,2,native english teacher at epik english program...,Canada,500+,NaN,education,0.329277,<NA>,Canada,500,0.312640,0.330935,mid
2,3,aspiring human resources professional,"Raleigh-Durham, North Carolina Area",44,NaN,human resources,0.773061,Raleigh-Durham,North Carolina Area,44,0.296837,0.686717,junior
3,4,people development coordinator at ryan,"Denton, Texas",500+,NaN,human resources,0.441673,Denton,Texas,500,0.424958,0.344441,mid
4,5,advisory board member at celal bayar university,"İzmir, Türkiye",500+,NaN,education,0.251772,İzmir,Türkiye,500,0.435025,0.397959,mid
5,6,aspiring human resources specialist,Greater New York City Area,1,NaN,human resources,0.779432,<NA>,Greater New York City Area,1,0.293982,0.596761,junior
6,7,student at humber college and aspiring human r...,Canada,61,NaN,human resources,0.601459,<NA>,Canada,61,0.282041,0.535022,junior
7,8,hr senior specialist,San Francisco Bay Area,500+,NaN,human resources,0.552003,<NA>,San Francisco Bay Area,500,0.743612,0.564974,senior
8,9,student at humber college and aspiring human r...,Canada,61,NaN,human resources,0.601459,<NA>,Canada,61,0.282041,0.535022,junior
9,10,seeking human resources hris and generalist po...,Greater Philadelphia Area,500+,NaN,human resources,0.724941,<NA>,Greater Philadelphia Area,500,0.328203,0.350675,mid


In [187]:
#identifying seniority levels in job titles
seniority_levels = ["Junior", "Mid", "Senior"]

#people with higher connections (500+) are more likely to be senior level.
#use words in the job title to help identify seniority levels. E.g. Senior, aspiring, student


senior_titles = [
    "senior professional",
    "experienced professional",
    "lead professional",
    "team lead",
    "manager",
    "senior manager",
    "director",
    "principal",
    "head of department",
    "executive"
]

junior_titles = [
    "junior",
    "junior professional",
    "intern",
    "student",
    "entry level",
    "entry level professional",
    "graduate",
    "recent graduate",
    "undergraduate",
    "undergraduate student",
    "undergraduate researcher",
    "trainee",
    "aspiring",
    "assisstant",
    "aspiring professional",    
]
model = SentenceTransformer("all-MiniLM-L6-v2")

senior_embeddings = model.encode(senior_titles, normalize_embeddings=True)
junior_embeddings = model.encode(junior_titles, normalize_embeddings=True)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5401.04it/s]


In [188]:
import numpy as np

job_embeddings = model.encode(df["job_title"].fillna("").tolist(), normalize_embeddings=True)

senior_scores = job_embeddings @ senior_embeddings.T
junior_scores = job_embeddings @ junior_embeddings.T

senior_similarity = senior_scores.max(axis=1)
junior_similarity = junior_scores.max(axis=1)

#add connections as considerations for seniority
df["connections"] = (df["connection"].str.replace("+", "", regex=False).astype("Int64"))
senior_connection_bonus = np.select([df["connections"] >= 500, df["connections"] >= 200], [0.08, 0.02], default=0)
junior_connection_bonus = np.select([df["connections"] <= 100, df["connections"] < 500], [0.08, 0.02], default=0)


adjusted_senior_similarity = senior_similarity + senior_connection_bonus
adjusted_junior_similarity = junior_similarity + junior_connection_bonus

df["adjusted_senior_similarity"] = adjusted_senior_similarity
df["adjusted_junior_similarity"] = adjusted_junior_similarity
threshold = 0.50

df["seniority"] = np.select(
    [
        (adjusted_senior_similarity >= threshold) & (adjusted_senior_similarity > adjusted_junior_similarity),
        (adjusted_junior_similarity >= threshold) & (adjusted_junior_similarity > adjusted_senior_similarity)
    ],
    ["senior", "junior"],
    default="mid"
)

pd.set_option("display.max_rows", None)
df

,id,job_title,location,connection,fit,identified_job_title,job_title_similarity,city,region,connections,adjusted_senior_similarity,adjusted_junior_similarity,seniority
0,1,2019 ct bauer college of business graduate mag...,"Houston, Texas",85,NaN,human resources,0.433673,Houston,Texas,85,0.372038,0.507028,junior
1,2,native english teacher at epik english program...,Canada,500+,NaN,education,0.329277,<NA>,Canada,500,0.347373,0.330935,mid
2,3,aspiring human resources professional,"Raleigh-Durham, North Carolina Area",44,NaN,human resources,0.773061,Raleigh-Durham,North Carolina Area,44,0.525125,0.686717,junior
3,4,people development coordinator at ryan,"Denton, Texas",500+,NaN,human resources,0.441673,Denton,Texas,500,0.486484,0.344441,mid
4,5,advisory board member at celal bayar university,"İzmir, Türkiye",500+,NaN,education,0.251772,İzmir,Türkiye,500,0.452719,0.397959,mid
5,6,aspiring human resources specialist,Greater New York City Area,1,NaN,human resources,0.779432,<NA>,Greater New York City Area,1,0.440221,0.596761,junior
6,7,student at humber college and aspiring human r...,Canada,61,NaN,human resources,0.601459,<NA>,Canada,61,0.366960,0.535022,junior
7,8,hr senior specialist,San Francisco Bay Area,500+,NaN,human resources,0.552003,<NA>,San Francisco Bay Area,500,0.843822,0.564974,senior
8,9,student at humber college and aspiring human r...,Canada,61,NaN,human resources,0.601459,<NA>,Canada,61,0.366960,0.535022,junior
9,10,seeking human resources hris and generalist po...,Greater Philadelphia Area,500+,NaN,human resources,0.724941,<NA>,Greater Philadelphia Area,500,0.460369,0.350675,mid


In [71]:
import re

def normalize_location(location):
    if pd.isna(location):
        return ""

    location = str(location).lower().strip()

    # Remove common LinkedIn location terms
    location = re.sub(r"\bgreater\b", "", location)
    location = re.sub(r"\barea\b", "", location)

    # Normalize whitespace
    location = re.sub(r"\s+", " ", location)

    return location.strip()


def find_country(city, region, location_dict):
    # If both are missing, there is nothing to search
    if pd.isna(city) and pd.isna(region):
        return pd.NA

    # Build location using whichever values exist
    parts = []

    if pd.notna(city):
        parts.append(str(city))

    if pd.notna(region):
        parts.append(str(region))

    location = normalize_location(", ".join(parts))

    matches = []

    for known_location, country in location_dict.items():
        known_location = normalize_location(known_location)

        if known_location in location or location in known_location:
            matches.append((len(known_location), country))

    if not matches:
        return pd.NA

    # Prefer the longest/most specific match
    matches.sort(reverse=True)

    return matches[0][1]

In [ ]:
#come back to this later
cities = pd.read_csv("../data/cities500.txt", sep="\t", header=None, low_memory=False)

cities.columns = [
    "geoname_id",
    "name",
    "ascii_name",
    "alternate_names",
    "latitude",
    "longitude",
    "feature_class",
    "feature_code",
    "country_code",
    "cc2",
    "admin1_code",
    "admin2_code",
    "admin3_code",
    "admin4_code",
    "population",
    "elevation",
    "dem",
    "timezone",
    "modification_date"
]

countries = pd.read_csv("../data/countryInfo.txt", sep="\t", comment="#", header=None)

countries.columns = [
    "country_code",
    "iso3",
    "iso_numeric",
    "fips",
    "country",
    "capital",
    "area",
    "population",
    "continent",
    "tld",
    "currency_code",
    "currency_name",
    "phone",
    "postal_code_format",
    "postal_code_regex",
    "languages",
    "geoname_id",
    "neighbours",
    "equivalent_fips"
]



area_country_mapping = {
    "Greater New York City Area": "United States",
    "Greater Los Angeles Area": "United States",
    "Kanada": "Canada",
    # etc.
}

In [ ]:
#preprocessing location column

#two formats of location: "Region" and "City, Country"

has_city_country = df["location"].str.contains(",", na=False)
split_location = df.loc[has_city_country, "location"].str.split(",", n=1, expand=True)

df["city"] = pd.NA
df["region"] = pd.NA
df["country"] = pd.NA

df.loc[has_city_country, "city"] = split_location[0].str.strip()
df.loc[has_city_country, "region"] = split_location[1].str.strip()


is_region = ~df["location"].str.contains(",", na=False) & (df["location"] != "Kanada")
df.loc[is_region, "region"] = df.loc[is_region, "location"]

is_canada = df["location"].str.contains("Kanada", na=False)
is_usa = ~is_canada
df.loc[is_canada, "country"] = "Canada"
df.loc[is_usa, "country"] = "USA"

# handle all null values the same way
df["city"] = df["city"].astype("string")
df["country"] = df["country"].astype("string")
df["region"] = df["region"].astype("string")

df["location"].head(103)

0                           Houston, Texas
1                                   Kanada
2      Raleigh-Durham, North Carolina Area
3                            Denton, Texas
4                           İzmir, Türkiye
                      ...                 
98                  Las Vegas, Nevada Area
99                Cape Girardeau, Missouri
100    Raleigh-Durham, North Carolina Area
101             Greater New York City Area
102               Greater Los Angeles Area
Name: location, Length: 103, dtype: str

In [ ]:
"""to do:

identify professions
identify education classes
identify seniority

similarity metrics


"""